Retreive 20 records including the last 4 columns from the GCS's file generated from ex01.ipynb
Using "gemini-3.6-flash" (or a version it recommends), ask it to summarize the reports.

- Pre-requisite : Generate API keys through [Google AI Studio](https://aistudio.google.com/app/apikey)

In [1]:
import datetime
import json
import os

from dotenv import load_dotenv
from google import genai
from google.oauth2 import service_account
from google.cloud import storage

In [2]:
load_dotenv()

True

In [3]:
gemini_api_key = os.getenv("GEMINI_API_KEY")
service_account_key = os.getenv("GCP_SERVICE_ACCOUNT_KEY")
project_id = os.getenv("GCP_PROJECT_ID")
bucket_name = os.getenv("GCP_BUCKET_NAME")
file_name = f"sf_police_report/{datetime.date.today()}.json"

In [4]:
def retrieve_data_from_gcs(service_account_key: str,
                           project_id: str,
                           bucket_name: str,
                           file_name: str,
                           key_list: list
                           ) -> list:
    credentials = service_account.Credentials.from_service_account_file(service_account_key)
    client = storage.Client(project=project_id,
                            credentials=credentials)
    bucket = client.bucket(bucket_name)
    file = bucket.blob(file_name)
    content = json.loads(file.download_as_string())

    output = []
    for data in content:
        row = []
        
        for key in key_list:
            row.append(data.get(key, None))
        output.append(row)
    return output

In [5]:
key_list = ['incident_datetime', 'report_datetime', 'incident_code',
            'incident_category', 'incident_description', 'latitude',
            'longitude', 'police_district']
data = retrieve_data_from_gcs(service_account_key, 
                              project_id,
                              bucket_name,
                              file_name,
                              key_list)

In [6]:
filtered_data = [row[-4:] for row in data][:20]

In [7]:
# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [11]:
model_name = "gemini-3.6-flash"

In [12]:
prompt_content = f"There has been a police report\
on the following list of [description, lon, lat, district] : {filtered_data} recently.\
Summarize the reports"

In [13]:
response = client.models.generate_content(
    model=model_name,
    contents=prompt_content
)

In [14]:
response.text

'Here is a summary of the **20 police reports** provided:\n\n---\n\n### **1. Breakdown by Incident Type**\n\n* **Vehicle-Related Incidents (8 total):**\n  * **Vehicle Break-Ins / Theft from Vehicle (4):** 3 involving locked vehicles with losses over $950 (*Northern, Southern, Park*), and 1 from an unlocked vehicle (*Southern*).\n  * **Recovered Vehicles (2):** Both located outside San Francisco (*Out of SF*).\n  * **Stolen Vehicle (1):** Theft of a non-auto vehicle (*Park*).\n  * **Vehicle Arson (1):** Incendiary incident involving a vehicle (*Tenderloin*).\n\n* **Lost Property & Vandalism (5 total):**\n  * **Lost Property (4):** Reported across various locations (*Park, Bayview, Tenderloin, and Out of SF*).\n  * **Malicious Mischief / Vandalism (1):** Damage to property (*Taraval*).\n\n* **Public Order, Narcotics & Police Resistance (4 total):**\n  * **Evading/Resisting Police (2):** 1 case of reckless evasion (*Southern*) and 1 case of resisting/obstructing an officer (*Mission*).\n 

In [ ]:
# Note: This is an optional to display markdown strings.
from rich.console import Console
from rich.markdown import Markdown

console = Console()
md = Markdown(response.text)
console.print(md)


Here is a summary of the 20 police reports provided:                                                               

-------------------------------------------------------------------------------------------------------------------

1. Breakdown by Incident Type                                                                                      

 • Vehicle-Related Incidents (8 total):                                                                            
    • Vehicle Break-Ins / Theft from Vehicle (4): 3 involving locked vehicles with losses over $950 (Northern,     
      Southern, Park), and 1 from an unlocked vehicle (Southern).                                                  
    • Recovered Vehicles (2): Both located outside San Francisco (Out of SF).                                      
    • Stolen Vehicle (1): Theft of a non-auto vehicle (Park).                                                      
    • Vehicle Arson (1): Incendiary incident involving a vehicle (Tenderloin).                                     
 • Lost Property & Vandalism (5 total):                                                                            
    • Lost Property (4): Reported across various locations (Park, Bayview, Tenderloin, and Out of SF).             
    • Malicious Mischief / Vandalism (1): Damage to property (Taraval).                                            
 • Public Order, Narcotics & Police Resistance (4 total):                                                          
    • Evading/Resisting Police (2): 1 case of reckless evasion (Southern) and 1 case of resisting/obstructing an   
      officer (Mission).                                                                                           
    • Drug Sales (1): Sale of opiates (Tenderloin).                                                                
    • Miscellaneous Investigation (1): Non-specified investigation (Taraval).                                      
 • Crimes Against Persons & Fraud (3 total):                                                                       
    • Battery (2): Physical altercations reported in Northern and Park districts.                                  
    • False Personation (1): Identity fraud/impersonation (Park).                                                  

-------------------------------------------------------------------------------------------------------------------

2. Breakdown by District                                                                                           

 • Park (5 reports): Highest concentration of reported incidents (Battery, Stolen Vehicle, Theft from Vehicle,     
   False Personation, Lost Property).                                                                              
 • Southern (3 reports): Evading Police, Theft from Locked Vehicle, Theft from Unlocked Vehicle.                   
 • Tenderloin (3 reports): Opiate Sale, Vehicle Arson, Lost Property.                                              
 • Out of SF (3 reports): 2 Recovered Vehicles, 1 Lost Property report.                                            
 • Northern (2 reports): Battery, Theft from Locked Vehicle.                                                       
 • Taraval (2 reports): Vandalism, Miscellaneous Investigation.                                                    
 • Mission (1 report): Resisting Peace Officer.                                                                    
 • Bayview (1 report): Lost Property.                                                                              

-------------------------------------------------------------------------------------------------------------------

3. Data Quality & Location Details                                                                                 

 • Location Coordinates Available: 14 out of 20 records include precise latitude/longitude coordinates.            
 • Missing Coordinates: 6 records lack GPS data